# 2. Профили участников
Распределение по уровням, сколько людей уходит после первого соревнования,
сколько времени остаются активными.

In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load, TIERS

sns.set_theme(style="whitegrid")
users = load("Users")
users.shape

## Уровни (PerformanceTier)

In [ ]:
tiers = users["PerformanceTier"].map(TIERS).value_counts().reindex(TIERS.values())
tiers

In [ ]:
tiers.drop("Staff").plot(kind="bar", logy=True, figsize=(8, 4), title="Пользователи по уровням (лог. шкала)")
plt.savefig("../reports/figures/02_users_by_tier.png", bbox_inches="tight")
plt.show()

## Участие в соревнованиях
Считаем по командам: в скольких соревнованиях участвовал человек и когда.

In [ ]:
teams = load("Teams")[["Id", "CompetitionId", "ScoreFirstSubmittedDate"]]
members = load("TeamMemberships")[["TeamId", "UserId"]]
part = members.merge(teams, left_on="TeamId", right_on="Id")

per_user = part.groupby("UserId").agg(
    n_comps=("CompetitionId", "nunique"),
    first=("ScoreFirstSubmittedDate", "min"),
    last=("ScoreFirstSubmittedDate", "max"),
)
per_user["active_days"] = (per_user["last"] - per_user["first"]).dt.days
per_user["n_comps"].describe()

In [ ]:
# доля тех, кто участвовал только в одном соревновании
(per_user["n_comps"] == 1).mean()

In [ ]:
per_user.loc[per_user["n_comps"] > 1, "active_days"].plot(
    kind="hist", bins=50, figsize=(8, 4), title="Срок активности (дней), участники с 2+ соревнованиями"
)
plt.show()

TODO: связать срок активности с уровнем участника.